# Section 1 — Data Quality Checks

Three factual questions built on `hotel_bookings.csv`.  
Every filter is tied to a specific footnote from the assessment brief.

---

In [12]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

## Load Data

Two non-negotiable guards applied at load time:

- `keep_default_na=False` — preserves the literal string `'None'` in `customer_loyalty_tier`. Without this, pandas silently converts it to `NaN` and ~46% of tier data disappears **(Footnote 7)**.
- `review_rating` and `review_date` are nullable by design (empty when no review was left). They are converted separately using `errors='coerce'` so that empty strings do not corrupt the column dtype.
- Note: actual column name in this CSV is `checkin_date` (no underscore), not `check_in_date`.

In [13]:
df = pd.read_csv(
    '../hotel_bookings (1).csv',
    keep_default_na=False,
    parse_dates=['customer_signup_date', 'booking_date', 'checkin_date', 'checkout_date']
)

# Nullable columns: empty strings -> NaN / NaT
df['review_rating'] = pd.to_numeric(df['review_rating'], errors='coerce')
df['review_date']   = pd.to_datetime(df['review_date'], errors='coerce')

print(f'Loaded: {len(df):,} rows x {df.shape[1]} columns')
df.dtypes

Loaded: 12,000 rows x 28 columns


booking_id                        int64
customer_id                       int64
customer_name                       str
customer_segment                    str
customer_signup_date     datetime64[us]
customer_home_city                  str
customer_loyalty_tier               str
property_id                       int64
property_name                       str
property_city                       str
property_star_rating              int64
property_type                       str
property_total_rooms              int64
booking_date             datetime64[us]
checkin_date             datetime64[us]
checkout_date            datetime64[us]
room_type                           str
num_rooms                         int64
nights                            int64
booking_channel                     str
adr                             float64
discount_amount                 float64
coupon_code                         str
total_amount                    float64
payment_method                      str


In [14]:
# Quick sanity check: loyalty tier 'None' should be a string, not NaN
tier_counts = df['customer_loyalty_tier'].value_counts(dropna=False)
print('Loyalty tier distribution (None must appear as a string, not be missing):')
print(tier_counts)

Loyalty tier distribution (None must appear as a string, not be missing):
customer_loyalty_tier
None        5571
Silver      3135
Gold        2100
Platinum    1194
Name: count, dtype: int64


---

## A1 — Invalid Stays Count

**Footnote 1:** Rows where `checkout_date <= checkin_date` are data errors.  
A valid hotel stay requires at minimum one night, meaning `checkout_date > checkin_date` must hold.

In [15]:
invalid_stays = df[df['checkout_date'] <= df['checkin_date']]

print(f'A1 Answer — Invalid stays (checkout_date <= checkin_date): {len(invalid_stays):,}')
print()
invalid_stays[['booking_id', 'checkin_date', 'checkout_date', 'nights', 'booking_status']].head(10)

A1 Answer — Invalid stays (checkout_date <= checkin_date): 120



,booking_id,checkin_date,checkout_date,nights,booking_status
151,100151,2024-12-13,2024-12-13,4,Completed
172,100172,2024-08-09,2024-08-08,2,Completed
245,100245,2024-09-30,2024-09-29,3,Completed
264,100264,2024-11-22,2024-11-22,2,Completed
375,100375,2024-05-19,2024-05-19,1,Completed
441,100441,2024-11-02,2024-11-01,1,Cancelled
473,100473,2024-10-28,2024-10-27,2,Cancelled
482,100482,2024-03-22,2024-03-22,2,Completed
541,100541,2024-09-08,2024-09-07,4,Completed
594,100594,2024-06-26,2024-06-26,2,Cancelled


> **A1 Answer: 120 bookings** have `checkout_date <= checkin_date` and are treated as data errors.  
> These rows are excluded from any revenue, stay-duration, or occupancy calculation throughout the rest of the analysis.

---

## A2 — Review Rating: Range and Mean by Customer Segment

**Footnote 6:** `review_rating` uses **different scales per segment**:
- `Individual` and `Group` — scale **1 to 5**
- `Corporate` — scale **1 to 10**

Comparing means across segments without normalization will make Corporate appear artificially higher-rated.  
We show **raw stats** first (to expose the problem), then **normalized stats** (Corporate / 2 to bring everything onto a 1–5 scale).  

Only rows with an actual review are included — NaN `review_rating` rows are dropped.

In [16]:
reviewed = df.dropna(subset=['review_rating'])

raw_stats = (
    reviewed
    .groupby('customer_segment')['review_rating']
    .agg(Min='min', Max='max', Mean='mean', Reviews='count')
    .round(2)
)

print('RAW stats — scale mismatch makes cross-segment mean comparison invalid:')
raw_stats

RAW stats — scale mismatch makes cross-segment mean comparison invalid:


,Min,Max,Mean,Reviews
customer_segment,,,,
Corporate,3.0,10.0,7.25,1287
Group,1.0,5.0,3.77,590
Individual,1.0,5.0,3.77,3269


In [17]:
# Normalize Corporate to 1-5 scale
reviewed_norm = reviewed.copy()
corp_mask = reviewed_norm['customer_segment'] == 'Corporate'
reviewed_norm.loc[corp_mask, 'review_rating'] = reviewed_norm.loc[corp_mask, 'review_rating'] / 2

norm_stats = (
    reviewed_norm
    .groupby('customer_segment')['review_rating']
    .agg(Min='min', Max='max', NormalizedMean='mean', Reviews='count')
    .round(2)
)

print('NORMALIZED stats — Corporate / 2, all segments now on 1-5 scale:')
norm_stats

NORMALIZED stats — Corporate / 2, all segments now on 1-5 scale:


,Min,Max,NormalizedMean,Reviews
customer_segment,,,,
Corporate,1.5,5.0,3.62,1287
Group,1.0,5.0,3.77,590
Individual,1.0,5.0,3.77,3269


**A2 Answer — Computed values:**

| Segment | Raw Min | Raw Max | Raw Mean | Normalized Mean (÷2 for Corporate) | Reviews |
|---|---|---|---|---|---|
| Corporate | 3.0 | 10.0 | 7.25 | 3.62 | 1,287 |
| Group | 1.0 | 5.0 | 3.77 | 3.77 | 590 |
| Individual | 1.0 | 5.0 | 3.77 | 3.77 | 3,269 |

> **1-line implication:** Corporate ratings are on a 1–10 scale while Individual and Group are on 1–5 — the raw Corporate mean (7.25) appears nearly double the others (3.77) purely due to the scale difference, not genuine satisfaction gap; after normalizing, all three segments are within 0.15 points of each other.

---

## A3 — Realized Revenue for Luxury Properties

Three footnotes must be applied **before** summing `total_amount`:

| Footnote | Issue | Action |
|---|---|---|
| **FN1** | Invalid stays (`checkout_date <= checkin_date`) | Exclude |
| **FN3** | Zero-room bookings (`num_rooms = 0`) | Exclude |
| **FN8** | Only `Completed` bookings count as realized revenue | Keep Completed only |

Two additional footnotes are data quality flags that do **not** change the revenue number but should be reported:
- **FN2** — bookings dated before customer signup date (temporal integrity violation)
- **FN5** — Cancelled bookings carrying a review rating (logically impossible)

After cleaning, filter `property_type == 'Luxury'` and sum `total_amount`.

In [18]:
# FN2 and FN5 — flagged but not dropped for revenue
fn2_count = (df['booking_date'] < df['customer_signup_date']).sum()
fn5_count = ((df['booking_status'] == 'Cancelled') & df['review_rating'].notna()).sum()

print(f'FN2 — Bookings dated before customer signup  : {fn2_count:,} rows')
print(f'FN5 — Cancelled bookings with a review       : {fn5_count:,} rows')

FN2 — Bookings dated before customer signup  : 163 rows
FN5 — Cancelled bookings with a review       : 50 rows


In [19]:
# Apply footnote filters in sequence, showing row count at each step
step1 = df[df['checkout_date'] > df['checkin_date']]       # FN1
step2 = step1[step1['num_rooms'] > 0]                       # FN3
step3 = step2[step2['booking_status'] == 'Completed']       # FN8
luxury = step3[step3['property_type'] == 'Luxury']          # target

realized_revenue = luxury['total_amount'].sum()

print(f'Starting rows (full dataset)              : {len(df):,}')
print(f'After FN1 (valid stays only)              : {len(step1):,}')
print(f'After FN3 (num_rooms > 0)                 : {len(step2):,}')
print(f'After FN8 (Completed only)                : {len(step3):,}')
print(f'After Luxury filter                       : {len(luxury):,}')
print()
print(f'Realized Revenue (Luxury) : Rs. {realized_revenue:,.2f}')

Starting rows (full dataset)              : 12,000
After FN1 (valid stays only)              : 11,880
After FN3 (num_rooms > 0)                 : 11,821
After FN8 (Completed only)                : 9,198
After Luxury filter                       : 1,063

Realized Revenue (Luxury) : Rs. 90,694,052.93


In [20]:
# Show what the naive (no-filter) sum looks like vs. realized
naive             = df[df['property_type'] == 'Luxury']['total_amount'].sum()
overstatement     = naive - realized_revenue
overstatement_pct = (overstatement / naive) * 100

print(f'Naive sum (no footnote filters)  : Rs. {naive:,.2f}')
print(f'Realized revenue (with filters)  : Rs. {realized_revenue:,.2f}')
print(f'Overstatement                    : Rs. {overstatement:,.2f}  ({overstatement_pct:.1f}% of naive total)')

Naive sum (no footnote filters)  : Rs. 117,431,484.47
Realized revenue (with filters)  : Rs. 90,694,052.93
Overstatement                    : Rs. 26,737,431.54  (22.8% of naive total)


**A3 Answer — Computed values:**

| | Amount |
|---|---|
| Naive sum (no filters) | Rs. 1,17,43,148.47 |
| **Realized revenue (FN1 + FN3 + FN8 applied)** | **Rs. 90,69,405.293** |
| Overstatement without filters | Rs. 26,73,743.154 (22.8% of naive) |

> Realized revenue for `property_type = Luxury` is **Rs. 90,694,052.93**.  
> The naive sum overstates it by 22.8% — because Cancelled and No-Show bookings carry a `total_amount` that was never collected (FN8). This is the single biggest source of inflation.

---

## Section 1 — Final Answers Summary

| Question | Answer |
|---|---|
| **A1 — Invalid stays** | **120 bookings** with `checkout_date <= checkin_date` |
| **A2 — Rating implication** | Corporate raw mean is 7.25 (on 1–10) vs. 3.77 for others (on 1–5). After normalizing (Corporate ÷ 2), all segments fall between 3.62–3.77 — scale mismatch, not a real satisfaction difference |
| **A3 — Luxury realized revenue** | **Rs. 90,694,052.93** after applying FN1 + FN3 + FN8 (naive sum overstates by 22.8%) |